# Day 16：大模型 API 核心参数详解：采样控制、对话角色协议与流式打字机（Jupyter Notebook 实战手册）

> **所属阶段**：第 2 阶段 · 第二阶段：大模型核心原理、Prompt工程与全栈 Web API ｜ **第 3 周**：第3周：大模型底层原理、Prompt 工程与工具调用实战  
> **GitLab 工单**：`STARMIND-016` ｜ **开发分支**：`feat/day16-llm-api-params-stream`  
> **配套 HTML5 课件**：`courseware/day16/index.html` ｜ **独立版本快照**：`versions/day16/`

---

## 🎙️ 老陈开场旁白

早啊各位！昨天大家把大模型的「内脏」看明白了，今天咱们来拧它的「控制旋钮」。调用 OpenAI / DeepSeek / 通义千问的 Chat Completions 接口时，很多人只传 `model` 和 `messages`，其余参数全靠默认，结果做金融数据提取时 `temperature=1.0` 给你编造数字，做营销文案时 `temperature=0.0` 写得像政府公文，用户还投诉「页面转圈 10 秒没反应」。今天咱们亲手把 `temperature`、`top_p`、`max_tokens`、`frequency_penalty` 的底层算法用 Python 还原出来，再把 `stream=True` 的流式打字机和 TTFT 首字延迟统计跑通，让你调参不再靠玄学！

### 承上启下
- **昨日基线**：昨天（Day 15）我们摸清了 Transformer 注意力机制与 Token 成本核算，把财务预算账算明白了。今天林产品提了个新问题：同样一个模型，为什么客服问答有时胡言乱语、有时又像复读机一样车轱辘话？而且用户反馈每次要干等 8 秒才蹦出一整段字，体验太差了！
- **明日预告**：明天（Day 17）我们将系统进入 Prompt 工程基础，掌握提示词四要素、Zero/Few-shot、XML 安全分隔符与 JSON 结构化约束，搭建星辰智脑 10 大企业场景模板库。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day16'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-016）

星辰智脑内测期间，客服与文案两个业务线共用一套默认 API 参数，导致客服工单提取偶尔出现幻觉字段，文案生成又过于死板；同时同步阻塞接口导致长回答等待超过 6 秒，用户频繁刷新页面造成重复扣费。

**今日迭代目标：**
- 实现纯 Python 采样参数模拟器，直观演示 `temperature`、`top_p`、`frequency_penalty` 如何改变下一 Token 概率分布。
- 封装标准 `system` / `user` / `assistant` 三角色消息校验与参数合法性检查。
- 实现 `stream=True` 流式打字机引擎与多组参数对比实验报告生成器。

**验收标准：**
- 温度趋近 0 时，采样结果 100% 锁定概率最高的 Token（贪婪解码）；温度升高时概率分布明显趋平。
- 核采样 `top_p` 过滤后剩余候选词的原始累积概率达到阈值且重新归一化之和为 1.0。
- 流式客户端能按 OpenAI SSE chunk 结构逐步产出增量文本、准确记录 TTFT 耗时，并严格遵守 `max_tokens` 上限截断。

---

## 📚 课堂核心技术笔记

### 1. 1. Temperature 与 Top-P 的本质区别与配合法则

- **Temperature（温度，范围 0.0 ~ 2.0）**：直接除以 Logits。`T=0` 等价于贪婪搜索（永远选最高分）；`T<0.3` 回答高度稳定可复现；`T=0.7~1.0` 表达更丰富；`T>1.5` 极易产生乱码胡话。
- **Top-P（核采样，范围 0.0 ~ 1.0）**：把候选词按概率从高到低排序，累加概率刚达到 `top_p`（例如 0.9）时立刻一刀切掉剩余的极低概率长尾词，避免模型抽到离谱词。
- **官方最佳实践**：OpenAI 与 DeepSeek 文档均建议「调整 `temperature` 或 `top_p` 其中之一即可，不要同时大幅调整两者」。

> **⚠️ 老陈避坑**：常见坑：以为 `temperature=0` 就 100% 绝对每次输出一模一样。在 GPU 分布式 MoE（如 DeepSeek-V3）推理中，由于浮点数并行累加顺序和批处理路由波动，`temperature=0` 仍可能有极微小的非确定性，测试时需结合 `seed` 或结构化断言。

### 2. 2. Frequency Penalty、Presence Penalty 与 Max Tokens

- **frequency_penalty（-2.0 ~ 2.0）**：按某个词在已生成文本中**出现的次数**成比例扣分，专门治「模型陷入复读机死循环」。
- **presence_penalty（-2.0 ~ 2.0）**：只要某个词**出现过一次**就扣固定分，鼓励模型引入新话题。
- **max_tokens**：硬性限制单次回答最多生成的 Token 数，既防账单失控，也能保护前端排版。如果回答因为触达 `max_tokens` 被截断，响应里的 `finish_reason` 会变成 `'length'` 而不是 `'stop'`！

> **⚠️ 老陈避坑**：线上高频 Bug：代码里只取 `choices[0].message.content` 却不检查 `finish_reason`，结果 JSON 回答被 `max_tokens` 从中间一刀腰斩，下游 `json.loads()` 直接崩溃！

### 3. 3. system / user / assistant 三角色消息设计

主流 Chat API 都是无状态的，多轮对话靠每次请求把完整 `messages` 列表传给模型：`system` 位于开头定义全局规则与语气；`user` 与 `assistant` 交替出现构成历史记忆。如果前端把用户输入直接拼进 `system`，会导致极严重的提示词注入风险。

> **⚠️ 老陈避坑**：安全红线：永远不要信任前端传入的 `role='system'` 消息，系统提示词必须由后端服务端强制注入并锁定在列表首位。


## 💻 实操步骤 1：步骤一：实现采样参数校验与 Softmax/Temperature/Top-P/Penalty 数学模拟器

- **目标文件**：`versions/day16/src/day16_sampling_simulator.py`
- **老陈代码旁白**：老陈带大家写 `src/day16_sampling_simulator.py`。咱们把 OpenAI 规范的 `GenerationConfig` 参数校验做严实，然后用纯 Python 还原一套完整的 `apply_sampling_pipeline`，让你用肉眼看到 `temperature=0.1` 和 `temperature=1.2` 对概率分布产生的剧烈影响。


In [ ]:
"""
StarMind Day 16 - 步骤一：大模型 API 核心采样参数校验与数学模拟器
完整还原 frequency_penalty -> temperature -> softmax -> top_p 核采样管线。
"""

import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple


@dataclass
class GenerationConfig:
    """大模型生成参数配置（对齐 OpenAI / DeepSeek Chat Completions 规范）。"""
    temperature: float = 0.7
    top_p: float = 0.9
    max_tokens: int = 512
    frequency_penalty: float = 0.0
    presence_penalty: float = 0.0
    stream: bool = True

    def validate(self) -> None:
        """在发往模型网关前执行严格的参数边界校验。"""
        if not (0.0 <= self.temperature <= 2.0):
            raise ValueError(f"temperature 必须在 [0.0, 2.0] 之间，当前值: {self.temperature}")
        if not (0.0 < self.top_p <= 1.0):
            raise ValueError(f"top_p 必须在 (0.0, 1.0] 之间，当前值: {self.top_p}")
        if self.max_tokens < 1 or self.max_tokens > 32768:
            raise ValueError(f"max_tokens 必须在 [1, 32768] 之间，当前值: {self.max_tokens}")
        if not (-2.0 <= self.frequency_penalty <= 2.0):
            raise ValueError(f"frequency_penalty 必须在 [-2.0, 2.0] 之间")
        if not (-2.0 <= self.presence_penalty <= 2.0):
            raise ValueError(f"presence_penalty 必须在 [-2.0, 2.0] 之间")


# 企业常用场景预设参数
SCENE_PRESETS: Dict[str, GenerationConfig] = {
    "extraction": GenerationConfig(temperature=0.0, top_p=1.0, max_tokens=512, frequency_penalty=0.0),
    "customer_service": GenerationConfig(temperature=0.3, top_p=0.9, max_tokens=800, frequency_penalty=0.2),
    "creative_copy": GenerationConfig(temperature=0.9, top_p=0.95, max_tokens=1200, frequency_penalty=0.5),
}


def compute_token_probabilities(
    logits: Dict[str, float],
    config: GenerationConfig,
    token_counts: Dict[str, int] | None = None,
) -> Dict[str, float]:
    """
    执行完整的采样概率变换：
    1. 施加 frequency_penalty 与 presence_penalty
    2. 施加 temperature 缩放（T=0 时退化为贪婪独热分布）
    3. Softmax 归一化
    4. Top-P 核采样截断并重新归一化
    """
    config.validate()
    if not logits:
        raise ValueError("logits 候选词字典不能为空")

    counts = token_counts or {}
    adjusted: Dict[str, float] = {}
    for token, score in logits.items():
        cnt = counts.get(token, 0)
        penalized = score - cnt * config.frequency_penalty - (1 if cnt > 0 else 0) * config.presence_penalty
        adjusted[token] = penalized

    # 温度为 0：贪婪解码，直接把 100% 概率赋予最高分 Token
    if config.temperature == 0.0:
        best_token = max(adjusted.items(), key=lambda kv: kv[1])[0]
        return {best_token: 1.0}

    # 温度缩放 + 数值稳定 Softmax
    scaled = {tok: val / config.temperature for tok, val in adjusted.items()}
    max_val = max(scaled.values())
    exps = {tok: math.exp(val - max_val) for tok, val in scaled.items()}
    sum_exp = sum(exps.values())
    probs = {tok: e / sum_exp for tok, e in exps.items()}

    # Top-P 核采样：按概率降序累加，保留达到 top_p 阈值的核心候选集合
    sorted_items: List[Tuple[str, float]] = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)
    kept: Dict[str, float] = {}
    cumulative = 0.0
    for tok, p in sorted_items:
        kept[tok] = p
        cumulative += p
        if cumulative >= config.top_p:
            break

    # 对截断后的候选池重新归一化，保证概率之和严格为 1.0
    kept_sum = sum(kept.values())
    return {tok: p / kept_sum for tok, p in kept.items()}


def sample_next_token(
    logits: Dict[str, float],
    config: GenerationConfig,
    token_counts: Dict[str, int] | None = None,
    seed: int | None = None,
) -> Tuple[str, Dict[str, float]]:
    """按计算后的概率分布抽样一个 Token，返回 (选中Token, 最终概率分布)。"""
    probs = compute_token_probabilities(logits, config, token_counts)
    rng = random.Random(seed)
    tokens = list(probs.keys())
    weights = list(probs.values())
    chosen = rng.choices(tokens, weights=weights, k=1)[0]
    return chosen, probs


if __name__ == "__main__":
    demo_logits = {"稳定": 4.2, "高效": 3.8, "智能": 3.5, "神奇": 1.5, "离谱": 0.2}
    for name, cfg in [
        ("低温严谨 T=0.1", GenerationConfig(temperature=0.1, top_p=0.9)),
        ("高温创意 T=1.2", GenerationConfig(temperature=1.2, top_p=0.9)),
        ("复读惩罚 FP=1.5", GenerationConfig(temperature=0.7, top_p=0.9, frequency_penalty=1.5)),
    ]:
        dist = compute_token_probabilities(demo_logits, cfg, token_counts={"稳定": 2})
        pretty = {k: round(v, 4) for k, v in dist.items()}
        print(f"[{name}] -> {pretty}")


### 🔍 步骤 1 代码深度解析

1. `GenerationConfig.validate()` 把参数校验前置在客户端，避免无效请求浪费网络 RTT。
2. `compute_token_probabilities` 完整实现了真实推理引擎（如 vLLM/llama.cpp）的采样链：先惩罚、再温控、后 Top-P 截断。
3. `sample_next_token` 接收独立 `random.Random(seed)` 实例，不污染全局随机状态，确保 CI 自动化测试 100% 稳定可复现。

**预期运行输出参考：**
```text
[低温严谨 T=0.1] -> {'稳定': 0.982}
[高温创意 T=1.2] -> {'稳定': 0.3712, '高效': 0.2659, '智能': 0.207, '神奇': 0.0391, ...}
[复读惩罚 FP=1.5] -> {'高效': 0.5621, '智能': 0.3663, '稳定': 0.0716}
```


## 💻 实操步骤 2：步骤二：实现多角色对话协议、stream=True 流式打字机与参数对比实验

- **目标文件**：`versions/day16/src/day16_stream_client.py`
- **老陈代码旁白**：接着写 `src/day16_stream_client.py`。这里我们把 `system/user/assistant` 角色协议校验、`stream=True` SSE Chunk 迭代器、`max_tokens` 截断与 `finish_reason` 标记全部落下来，并且内置一个参数对比实验运行器，一键输出不同参数下的回答风格与性能指标。


In [ ]:
"""
StarMind Day 16 - 步骤二：多角色对话协议、stream=True 流式打字机与参数对比实验
"""

import time
from dataclasses import dataclass
from typing import Dict, Generator, List

from src.day16_sampling_simulator import GenerationConfig, SCENE_PRESETS


VALID_ROLES = {"system", "user", "assistant"}


@dataclass
class StreamChunk:
    """模拟 OpenAI ChatCompletionChunk 增量结构。"""
    chunk_index: int
    delta_content: str
    finish_reason: str | None = None
    elapsed_ms: float = 0.0


@dataclass
class ExperimentResult:
    """单组参数对比实验汇总。"""
    profile_name: str
    temperature: float
    max_tokens: int
    output_text: str
    chunk_count: int
    ttft_ms: float
    total_ms: float
    finish_reason: str


class StarMindStreamClient:
    """
    星辰智脑离线/仿真流式对话客户端。
    完整遵循 system/user/assistant 协议与 stream=True 增量输出语义。
    """

    def validate_messages(self, messages: List[Dict[str, str]]) -> None:
        """校验对话消息列表是否符合标准角色协议。"""
        if not messages:
            raise ValueError("messages 不能为空")
        for idx, msg in enumerate(messages):
            role = msg.get("role", "")
            content = msg.get("content", "")
            if role not in VALID_ROLES:
                raise ValueError(f"第 {idx} 条消息角色非法: {role}，仅允许 {VALID_ROLES}")
            if not isinstance(content, str) or not content.strip():
                raise ValueError(f"第 {idx} 条消息 content 不能为空")
            if role == "system" and idx != 0:
                raise ValueError("安全规范：system 角色仅允许出现在消息列表首位（index=0）")

    def _synthesize_reply(self, messages: List[Dict[str, str]], config: GenerationConfig) -> str:
        """根据角色指令和温度参数生成确定性风格的模拟企业回复。"""
        last_user = next(
            (m["content"] for m in reversed(messages) if m["role"] == "user"),
            "",
        )
        if config.temperature <= 0.2:
            return f"【精准模式】针对「{last_user}」，建议按星辰数科 SOP-01 执行：1. 核对单号；2. 确认权限；3. 归档工单。"
        if config.temperature <= 0.6:
            return f"您好！关于「{last_user}」，星辰智脑为您梳理了清晰步骤：先核实基础信息，再提交审批流，预计 10 分钟内完成。"
        return f"哇，说到「{last_user}」这个话题太有意思啦！星辰智脑灵光一闪，为您准备了三套脑洞大开的创意方案供您挑选！"

    def stream_chat(
        self,
        messages: List[Dict[str, str]],
        config: GenerationConfig,
        chunk_size: int = 4,
        simulate_delay_ms: float = 0.0,
    ) -> Generator[StreamChunk, None, None]:
        """
        以生成器方式模拟 stream=True 流式输出：
        - 严格遵守 max_tokens（此处按字符分片近似）上限，超长时标记 finish_reason='length'
        - 正常结束时标记 finish_reason='stop'
        """
        self.validate_messages(messages)
        config.validate()

        full_text = self._synthesize_reply(messages, config)
        truncated = len(full_text) > config.max_tokens
        effective_text = full_text[: config.max_tokens]

        start = time.perf_counter()
        idx = 0
        for pos in range(0, len(effective_text), max(1, chunk_size)):
            if simulate_delay_ms > 0:
                time.sleep(simulate_delay_ms / 1000.0)
            piece = effective_text[pos : pos + max(1, chunk_size)]
            elapsed = round((time.perf_counter() - start) * 1000, 3)
            yield StreamChunk(chunk_index=idx, delta_content=piece, finish_reason=None, elapsed_ms=elapsed)
            idx += 1

        final_elapsed = round((time.perf_counter() - start) * 1000, 3)
        yield StreamChunk(
            chunk_index=idx,
            delta_content="",
            finish_reason="length" if truncated else "stop",
            elapsed_ms=final_elapsed,
        )

    def run_param_experiment(
        self,
        messages: List[Dict[str, str]],
        presets: Dict[str, GenerationConfig] | None = None,
    ) -> List[ExperimentResult]:
        """对同一组对话运行多组参数预设，产出结构化对比实验报告。"""
        active_presets = presets or SCENE_PRESETS
        results: List[ExperimentResult] = []
        for name, cfg in active_presets.items():
            pieces: List[str] = []
            ttft = 0.0
            total = 0.0
            finish = "stop"
            chunks = 0
            for chunk in self.stream_chat(messages, cfg, chunk_size=6, simulate_delay_ms=0.0):
                if chunks == 0:
                    ttft = chunk.elapsed_ms
                if chunk.delta_content:
                    pieces.append(chunk.delta_content)
                if chunk.finish_reason is not None:
                    finish = chunk.finish_reason
                total = chunk.elapsed_ms
                chunks += 1
            results.append(
                ExperimentResult(
                    profile_name=name,
                    temperature=cfg.temperature,
                    max_tokens=cfg.max_tokens,
                    output_text="".join(pieces),
                    chunk_count=chunks,
                    ttft_ms=ttft,
                    total_ms=total,
                    finish_reason=finish,
                )
            )
        return results


if __name__ == "__main__":
    client = StarMindStreamClient()
    msgs = [
        {"role": "system", "content": "你是杭州星辰数科企业 AI 助手。"},
        {"role": "user", "content": "新员工如何申请内网 VPN 权限？"},
    ]
    for exp in client.run_param_experiment(msgs):
        print(f"[{exp.profile_name:16s} | T={exp.temperature}] ({exp.finish_reason}) {exp.output_text}")


### 🔍 步骤 2 代码深度解析

1. `validate_messages` 强制要求 `system` 角色只能出现在首位，这是企业级对话接口防越权注入的重要习惯。
2. `stream_chat` 使用 Python `Generator` 惰性产出 `StreamChunk`，完整复现了 OpenAI SSE 流式响应中「若干 delta 块 + 最后一个携带 `finish_reason` 的结束块」的真实协议。
3. `run_param_experiment` 一次性对比多种场景参数并记录 `ttft_ms`（首字延迟）与 `finish_reason`，方便团队量化评审调参效果。

**预期运行输出参考：**
```text
[extraction       | T=0.0] (stop) 【精准模式】针对「新员工如何申请内网 VPN 权限？」...
[customer_service | T=0.3] (stop) 您好！关于「新员工如何申请内网 VPN 权限？」...
[creative_copy    | T=0.9] (stop) 哇，说到「新员工如何申请内网 VPN 权限？」...
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day16/tests/test_day16.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 16 自动化单元测试：验证采样参数数学变换、角色校验与流式打字机协议。"""

import pytest
from src.day16_sampling_simulator import (
    GenerationConfig,
    compute_token_probabilities,
    sample_next_token,
)
from src.day16_stream_client import StarMindStreamClient


def test_temperature_zero_is_greedy():
    logits = {"A": 5.0, "B": 4.9, "C": 1.0}
    cfg = GenerationConfig(temperature=0.0, top_p=1.0)
    probs = compute_token_probabilities(logits, cfg)
    assert probs == {"A": 1.0}
    token, _ = sample_next_token(logits, cfg, seed=42)
    assert token == "A"


def test_top_p_and_frequency_penalty():
    logits = {"重复词": 4.0, "新词A": 3.5, "新词B": 3.0, "长尾词": -5.0}
    cfg = GenerationConfig(temperature=0.8, top_p=0.85, frequency_penalty=1.5)
    probs = compute_token_probabilities(logits, cfg, token_counts={"重复词": 2})
    # 归一化之和严格为 1.0
    assert abs(sum(probs.values()) - 1.0) < 1e-6
    # 施加词频惩罚后，"新词A" 概率应反超 "重复词"
    assert probs.get("新词A", 0.0) > probs.get("重复词", 0.0)
    # 极低概率的长尾词应被 top_p 裁剪掉
    assert "长尾词" not in probs


def test_invalid_config_and_role_validation():
    with pytest.raises(ValueError):
        GenerationConfig(temperature=2.5).validate()
    with pytest.raises(ValueError):
        GenerationConfig(top_p=0.0).validate()

    client = StarMindStreamClient()
    with pytest.raises(ValueError):
        client.validate_messages([
            {"role": "user", "content": "hi"},
            {"role": "system", "content": "恶意中途插入 system"},
        ])


def test_stream_chat_truncation_and_experiment():
    client = StarMindStreamClient()
    msgs = [
        {"role": "system", "content": "你是星辰智脑助手"},
        {"role": "user", "content": "请介绍请假流程"},
    ]
    # 故意设置极小 max_tokens=10 验证 finish_reason='length'
    short_cfg = GenerationConfig(temperature=0.1, max_tokens=10)
    chunks = list(client.stream_chat(msgs, short_cfg, chunk_size=3))
    assembled = "".join(c.delta_content for c in chunks)
    assert len(assembled) == 10
    assert chunks[-1].finish_reason == "length"

    # 运行完整对比实验
    results = client.run_param_experiment(msgs)
    assert len(results) == 3
    assert all(r.finish_reason == "stop" for r in results)
    assert results[0].output_text != results[2].output_text

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：Top-K 采样过滤器实现

**题目要求：**
除了 Top-P，很多开源模型（如 Qwen / Llama）还支持 `top_k` 参数（仅保留分数最高的 K 个候选词）。请编写函数 `top_k_probs(logits, k, temperature=1.0)`，只保留前 K 个词并输出归一化概率字典。

**思路提示：** 先按 logit 降序切片取前 `k` 个，再做温度缩放和 Softmax。


In [ ]:
import math
from typing import Dict


def top_k_probs(logits: Dict[str, float], k: int, temperature: float = 1.0) -> Dict[str, float]:
    """Top-K 截断 + 温度 Softmax 概率计算。"""
    if k < 1:
        raise ValueError("k 必须 >= 1")
    if temperature <= 0:
        raise ValueError("temperature 必须 > 0")
    top_items = sorted(logits.items(), key=lambda kv: kv[1], reverse=True)[:k]
    scaled = {tok: score / temperature for tok, score in top_items}
    max_s = max(scaled.values())
    exps = {tok: math.exp(val - max_s) for tok, val in scaled.items()}
    total = sum(exps.values())
    return {tok: e / total for tok, e in exps.items()}


if __name__ == "__main__":
    print(top_k_probs({"猫": 4.0, "狗": 3.5, "鸟": 2.0, "石头": 0.1}, k=2, temperature=0.8))


**答案解析与评分标准：**

Top-K 固定保留词数，Top-P 动态适应概率分布陡峭度；在开源推理引擎中常先做 Top-K 再做 Top-P。


### 作业 2（企业实战）：流式输出中断续写器（Auto-Continue on Length）

**题目要求：**
当 `stream_chat` 返回 `finish_reason == 'length'` 时，自动把已生成内容拼为 `assistant` 消息、追加一条 `user: '请继续'` 并再次调用，最多续写 2 次，最终拼接返回完整文本。

**思路提示：** 循环调用 `StarMindStreamClient.stream_chat`，检查最后一个 chunk 的 `finish_reason`。


In [ ]:
from typing import Dict, List
from src.day16_sampling_simulator import GenerationConfig
from src.day16_stream_client import StarMindStreamClient


def chat_with_auto_continue(
    messages: List[Dict[str, str]],
    config: GenerationConfig,
    max_rounds: int = 2,
) -> str:
    """当输出因 max_tokens 被截断时自动续写。"""
    client = StarMindStreamClient()
    conv = list(messages)
    collected: List[str] = []

    for _ in range(max_rounds + 1):
        chunks = list(client.stream_chat(conv, config, chunk_size=5))
        part = "".join(c.delta_content for c in chunks)
        collected.append(part)
        if chunks[-1].finish_reason != "length":
            break
        conv.append({"role": "assistant", "content": part})
        conv.append({"role": "user", "content": "请接着上面的内容继续"})

    return "\n".join(collected)


if __name__ == "__main__":
    msgs = [{"role": "user", "content": "请详细介绍星辰智脑架构"}]
    print(chat_with_auto_continue(msgs, GenerationConfig(temperature=0.2, max_tokens=20), max_rounds=1))


**答案解析与评分标准：**

检查 `finish_reason` 是企业级长文档生成（如标书、周报）防截断的标准工程模式。


### 作业 3（架构进阶）：流式文本缓冲与按句推送适配器

**题目要求：**
在语音合成（TTS）或实时翻译场景中，不能每来 1 个字就调用下游接口，需要把流式 Token 缓存在内存里，每遇到中文句末标点（`。！？`）才立刻吐出一整句。请编写生成器 `buffer_by_sentence(chunks)`。

**思路提示：** 维护一个字符串缓冲区，不断追加 `chunk.delta_content`，用正则或循环按句末标点切分并 `yield`；流结束时把剩余缓冲区全部 `yield` 出去。


In [ ]:
from typing import Generator, Iterable
from src.day16_sampling_simulator import GenerationConfig
from src.day16_stream_client import StarMindStreamClient, StreamChunk

SENTENCE_ENDINGS = ("。", "！", "？")


def buffer_by_sentence(chunks: Iterable[StreamChunk]) -> Generator[str, None, None]:
    """将细粒度 Token 流聚合成完整句子流，适配实时 TTS/翻译下游。"""
    buf = ""
    for chunk in chunks:
        buf += chunk.delta_content
        while True:
            positions = [buf.find(p) for p in SENTENCE_ENDINGS if p in buf]
            if not positions:
                break
            cut = min(positions) + 1
            yield buf[:cut].strip()
            buf = buf[cut:]
    if buf.strip():
        yield buf.strip()


if __name__ == "__main__":
    client = StarMindStreamClient()
    stream = client.stream_chat(
        [{"role": "user", "content": "如何提升客服满意度？"}],
        GenerationConfig(temperature=0.4, max_tokens=200),
        chunk_size=3,
    )
    for sent in buffer_by_sentence(stream):
        print("整句推送 ->", sent)


**答案解析与评分标准：**

这是实时语音 Agent（ASR -> LLM -> TTS）降低端到端延迟的关键技巧：既不用等整段回答结束，又避免了逐字合成语音导致的断断续续。


---

## 🎯 老陈收尾复盘

今天大家亲手复现了温度缩放、核采样截断、词频惩罚和流式分片推送，是不是一下子觉得 API 文档里那些枯燥的参数活起来了？记住生产黄金法则：代码生成与数据提取用低温度（0.0~0.2），创意文案用中高温度（0.7~0.9），永远不要同时大幅乱调 `temperature` 和 `top_p`，面向用户的对话接口必须默认开启 `stream=True`！
